# 04 — Real Data Ingestion + Hybrid Pipeline

End-to-end pipeline: real + Kenya-context synthetic data → GNN training (both domains).

## What runs here

| Step | Data source | Pipeline output |
|------|-------------|-----------------|
| 1 | CISA KEV + FIRST EPSS (live API) | `VULNERABILITY_EVENT` rows |
| 2 | CIC-IDS2018 traffic (real CSV or auto-generated Kenya synthetic) | `DDOS_SIGNAL_EVENT` + `WEB_ATTACK_EVENT` rows |
| 3 | CAIDA DDoS traces (real CSV or auto-generated Kenya synthetic) | `DDOS_SIGNAL_EVENT` rows |
| 4 | Kenya procurement fraud synthetic seed | `graph_feature_snapshot` rows (`Wcorruption`) |
| 5 | Feature snapshot worker | updates `Wmid` and `Wcorruption` windows |
| 6 | Cyber GNN train | artifact saved to `/app/artifacts/gnn/` |
| 7 | Corruption GNN train | artifact saved to `/app/artifacts/gnn/` |

## Data flow
```
raw rows
  → app.integrations.real_data_pipeline normalizers
  → connector events
  → event_log + event_entity_index
  → graph_feature_snapshot (Wmid / Wcorruption)
  → GNN train/eval
```

## Real datasets (optional — pipeline uses Kenya synthetic if not present)

| Dataset | URL | Use |
|---------|-----|-----|
| CIC-IDS2018 | https://www.kaggle.com/datasets/solarmainframe/ids-intrusion-csv | DDoS + Web attacks |
| CAIDA DDoS 2007 | https://www.caida.org/catalog/datasets/ddos-20070804_dataset/ | Volumetric DDoS (registration required) |
| PaySim mobile money | https://www.kaggle.com/datasets/ntnu-testimon/paysim1 | M-Pesa fraud EDA |
| CISA KEV + EPSS | Live APIs (no download) | Vulnerability events ✅ already wired |

## Inputs you control

- `SOURCE_API_KEY` — use a seeded source key (default `safaricom-secret-key` works with `seed_default_sources()`)
- `ASSET_ID` — target asset ID for KEV vulnerability events
- `CIC_INPUT_FILE` — path to real CIC-IDS2018 CSV **or** `None` to auto-generate Kenya synthetic data
- `CAIDA_INPUT_FILE` — path to real CAIDA traces CSV **or** `None` to auto-generate Kenya synthetic data

**To use real CIC data:** download from Kaggle, unzip, set `CIC_INPUT_FILE = '/path/to/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv'`

**To use real CAIDA data:** register at caida.org, download `ddos-20070804`, convert to CSV, set `CAIDA_INPUT_FILE = '/path/to/caida_rows.csv'`

All generated synthetic files go into `notebooks/data/` which is git-ignored.

In [ ]:
import sys
from pathlib import Path

HERE = Path.cwd()
NOTEBOOKS_DIR = HERE if HERE.name == 'notebooks' else (HERE / 'notebooks')
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

from pipeline_bootstrap import (
    bootstrap_environment,
    check_notebook_prerequisites,
    repair_notebook_schema,
    event_type_counts_last_24h,
    ingest_kev_epss,
    ingest_traffic_file,
    run_feature_snapshots,
    seed_default_sources,
    train_gnn,
)
from seed_realistic_data import generate_cic_csv, generate_caida_csv

env = bootstrap_environment()
print("Environment:", env)

pre = check_notebook_prerequisites()
if not pre.get('ok'):
    print("Schema issues detected — auto-repairing...")
    print(repair_notebook_schema())

seed_default_sources()
print("Sources seeded.")

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
SOURCE_API_KEY = 'safaricom-secret-key'
ASSET_ID       = 'county-finance-db-01'

# CISA KEV + EPSS: always fetched live — no file needed
RUN_KEV = True

# CIC traffic: set to a real CIC-IDS2018 CSV path, or None for Kenya synthetic
CIC_INPUT_FILE = None    # e.g. '/data/cicids2018/Friday-DDos.csv'

# CAIDA traffic: set to a real CAIDA traces CSV path, or None for Kenya synthetic
CAIDA_INPUT_FILE = None  # e.g. '/data/caida/ddos-20070804-rows.csv'

# Corruption pipeline
RUN_CORRUPTION_SEED  = True   # seed Kenya procurement fraud patterns into DB
RUN_CORRUPTION_TRAIN = True   # train corruption GNN after seeding

In [ ]:
# ── Ingest cyber events (KEV + CIC + CAIDA) ───────────────────────────────────
results = []

if RUN_KEV:
    print("Ingesting CISA KEV + FIRST EPSS (live API)...")
    results.append(ingest_kev_epss(source_api_key=SOURCE_API_KEY, asset_id=ASSET_ID))
    print("  →", results[-1])

# CIC file is always available at this point (real or synthetic)
print("Ingesting CIC traffic data...")
results.append(
    ingest_traffic_file(
        dataset='cic',
        input_file=CIC_INPUT_FILE,
        source_api_key=SOURCE_API_KEY,
        service_id_prefix='kenya-infra',
        dataset_name='cic_ids2018_kaggle',
    )
)
print("  →", results[-1])

# CAIDA file is always available at this point (real or synthetic)
print("Ingesting CAIDA DDoS data...")
results.append(
    ingest_traffic_file(
        dataset='caida',
        input_file=CAIDA_INPUT_FILE,
        source_api_key=SOURCE_API_KEY,
        service_id_prefix='kenya-infra',
        dataset_name='caida_ddos',
    )
)
print("  →", results[-1])

results

In [ ]:
# ── Seed corruption domain data ────────────────────────────────────────────────
# Inserts Kenya procurement fraud patterns into graph_feature_snapshot
# (window_key="Wcorruption") so the corruption GNN can be trained.
# Families: Tender Cartel, Ghost Workers, Inflated Procurement,
#           Shell Company, FY-End Surge, Land Fraud

if RUN_CORRUPTION_SEED:
    print("Seeding Kenya corruption patterns...")
    from app.demo.synthetic_corruption_data import seed as seed_corruption
    corruption_seed_result = seed_corruption()
    print("Corruption seed complete:", corruption_seed_result)
else:
    corruption_seed_result = {}
    print("Corruption seed skipped (RUN_CORRUPTION_SEED=False)")

In [ ]:
# ── Build feature snapshots for both windows ──────────────────────────────────
# Wmid       → cyber GNN input (vulnerability + DDoS + web attack events)
# Wcorruption → corruption GNN input (already built by seed_corruption above,
#               re-run here to pick up any new events from the cyber ingest)

print("Building Wmid feature snapshots (cyber)...")
cyber_stats = run_feature_snapshots(window_keys=('Wmid',), max_entities=6_000)
print("  Wmid:", cyber_stats)

print("Building Wcorruption feature snapshots (corruption)...")
corruption_stats = run_feature_snapshots(window_keys=('Wcorruption',), max_entities=6_000)
print("  Wcorruption:", corruption_stats)

{'cyber': cyber_stats, 'corruption': corruption_stats}

In [ ]:
# ── Train Cyber GNN ───────────────────────────────────────────────────────────
print("Training Cyber GNN (window=Wmid, epochs=60)...")
cyber_train = train_gnn(domain='cyber', epochs=60)
print("Cyber GNN result:", cyber_train)

In [ ]:
# ── Train Corruption GNN ──────────────────────────────────────────────────────
if RUN_CORRUPTION_TRAIN:
    print("Training Corruption GNN (window=Wcorruption, epochs=60)...")
    corruption_train = train_gnn(domain='corruption', epochs=60)
    print("Corruption GNN result:", corruption_train)
else:
    corruption_train = {}
    print("Corruption GNN training skipped (RUN_CORRUPTION_TRAIN=False)")

In [ ]:
# ── Pipeline summary ──────────────────────────────────────────────────────────
summary = {
    'ingest':     results,
    'cyber_features':     cyber_stats,
    'corruption_features': corruption_stats,
    'cyber_gnn':     cyber_train,
    'corruption_gnn': corruption_train,
}

print("\n=== Pipeline complete ===")
for k, v in summary.items():
    print(f"  {k}: {v}")

summary

In [ ]:
feature_stats = run_feature_snapshots(window_keys=('Wmid',), max_entities=6000)
feature_stats


In [ ]:
train_result = train_gnn(domain='cyber', epochs=60)
train_result
